# Phase 3: Rhetorical Roles - GraphProp with LoRA

## Task
Rhetorical role labelling: every sentence of a legal document (Corporate Law and IT domains) gets one of 13 rhetorical-role labels (facts, arguments, ruling and so on). Scored with macro F1 over the labels.

Model: the pre-trained GraphProp model is frozen; LoRA is added to the linear layers in the blocks, and a sentence classifier is trained on top.

## Input
- Phase-3 data (CL and IT train / dev / test files and the label list); downloaded from Hugging Face if not attached (needs HF_TOKEN).
- GraphProp checkpoint and its tokenizer file.

## Flow of the notebook (from input to output)
1. Set the file paths and training choices.
2. Write the rhetorical-role pipeline (rr_core.py) and the fine-tuning methods (rr_peft.py).
3. Write the GraphProp model code.
4. Import the code, load the documents, tokenize every sentence and build the training settings.
5. Build the model: frozen GraphProp + LoRA + sentence classifier.
6. Train on the train documents and score on dev; then retrain on all documents and predict the test set (with optional rare-label boost and label smoothing over sentences).
7. Show the dev macro F1 of every setting.
8. Zip all outputs.

## Output
- Submission zips (the ones with full in the name come from the model trained on all documents).
- Dev scores and one zip with all outputs.

In [8]:
# Settings: file paths and training choices
CKPT = "/kaggle/input/datasets/preetpatel2315/graphporp-pt-f/legal_graphprop_random_final_random_90_10.pt"
TOK  = "/kaggle/input/datasets/preetpatel2315/tokenizer-graphporp-final-p2/tokenizer_graphprop.json"
HF_DATASET = "Exploration-Lab/CS-781-Capstone"

EPOCHS       = 15
BS           = 4
ACCUM        = 4
W            = 512
CTX          = "window"
HEAD_LR      = 2e-3
CLASS_POWER  = 0.5
LABEL_SMOOTH = 0.05
TAUS         = [0.0, 0.3, 0.6, 1.0]
DEV_SEEDS    = [0]
FULL_SEEDS   = [0]
SWA_LAST     = 3
GRU_HIDDEN   = 256
SNAP_EPOCHS  = [10, 12, 15]
VIT_LAMS     = [0.0, 0.5]
RARE_DUP     = 2

PEFT_LR      = 5e-4
R            = 16
ALPHA        = 32
LORA_DROPOUT = 0.05
HEAD_DROPOUT = 0.2

STAGES = ("dev", "full")
SMOKE = False
OUT = "/kaggle/working"

if SMOKE:
    EPOCHS, BS, ACCUM, W, CTX, HEAD_LR, CLASS_POWER, LABEL_SMOOTH, TAUS, DEV_SEEDS, FULL_SEEDS, SWA_LAST, GRU_HIDDEN, RARE_DUP, SNAP_EPOCHS, VIT_LAMS = 2, 2, 1, 128, "window", 2e-3, 0.5, 0.05, [0.0, 0.5], [0], [0], 1, 32, 2, [1], [0.0, 0.5]
    SMOKE_DOCS = 6
    PEFT_LR, R, ALPHA, LORA_DROPOUT, HEAD_DROPOUT = 2e-3, 4, 8, 0.05, 0.1

In [9]:
%%writefile rr_core.py
# Main rhetorical-role code: data, model, training (saved as rr_core.py)
import os, re, json, glob, math, time, copy, random, zipfile
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

REPO = "Exploration-Lab/CS-781-Capstone"
DATA_NAMES = ["CL_train.json", "CL_dev.json", "IT_train.json", "IT_dev.json", "label_vocab.json", "CL_test.json", "IT_test.json"]

# It takes name and roots (folders to search) as input, searches the input folders for a file and returns a file path (or None if not found).
def find_file(name, roots=("/kaggle/input", "/kaggle/working", ".")):
    for r in roots:
        hits = [h for h in glob.glob(os.path.join(r, "**", name), recursive=True) if "__MACOSX" not in h]
        if hits:
            return sorted(hits, key=len)[0]
    return None

# It takes dest (download folder) and repo (dataset name) as input, downloads the Phase-3 data if it is not attached and returns nothing.
def ensure_phase3_data(dest="/kaggle/working/phase3_data", repo=REPO):
    if all(find_file(n) for n in DATA_NAMES):
        print("Phase-3 files found locally:", os.path.dirname(find_file("CL_train.json"))); return
    from huggingface_hub import HfApi, hf_hub_download, login
    tok = None
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        tok = os.environ.get("HF_TOKEN")
    if not tok:
        import getpass
        tok = getpass.getpass("Hugging Face token (read access, dataset terms accepted): ")
    login(token=tok)
    files = HfApi().list_repo_files(repo, repo_type="dataset")
    os.makedirs(dest, exist_ok=True)
    for n in DATA_NAMES:
        if find_file(n, roots=(dest,)): continue
        c = [f for f in files if f == n or f.endswith("/" + n)]
        pref = [f for f in c if any(k in f.lower() for k in ("phase_3", "phase3", "_rr", "/rr"))]
        c = pref or c
        if not c: raise RuntimeError(f"{n} not found in {repo}. Files look like: {files[:15]}")
        if len(c) > 1: print(f"  note: {n} appears {len(c)} times {c[:4]}; using {sorted(c, key=len)[0]}")
        p = hf_hub_download(repo, sorted(c, key=len)[0], repo_type="dataset", local_dir=dest)
        print("  downloaded", n, "->", p)

# It takes path (file path) as input, loads the list of labels and returns a list of labels.
def load_vocab(path):
    v = json.load(open(path, encoding="utf-8"))
    if isinstance(v, dict):
        v = [k for k, _ in sorted(v.items(), key=lambda kv: kv[1])] if all(isinstance(x, int) for x in v.values()) else list(v.values())
    v = ["None" if x is None else str(x) for x in v]
    assert len(v) == len(set(v)), "label_vocab has duplicates"
    return v

# It takes path (file path) and domain (domain name) as input, loads one data file of documents and labels and returns a dictionary of documents.
def load_split(path, domain):
    raw = json.load(open(path, encoding="utf-8"))
    if isinstance(raw, dict) and isinstance(raw.get("text"), list) and not any(isinstance(v, dict) for v in raw.values()): items = [raw]
    elif isinstance(raw, dict): items = [({**v, "id": v.get("id", k)} if isinstance(v, dict) else {"id": k, "text": v}) for k, v in raw.items()]
    else: items = raw
    docs = {}
    for it in items:
        did = str(it["id"]); sents = [str(s) for s in it["text"]]; labs = it.get("labels")
        if labs is not None:
            labs = ["None" if l is None else str(l) for l in labs]
            assert len(labs) == len(sents), f"{path} doc {did}: {len(labs)} labels for {len(sents)} sentences"
        docs[f"{domain}|{did}"] = dict(id=did, domain=domain, sents=sents, labels=labs)
    return docs

# It takes labels (label names) as input, loads the train, dev and test documents and returns a dictionary with labels, train, dev and test.
def load_all(labels=None):
    vocab = load_vocab(find_file("label_vocab.json"))
    out = dict(labels=vocab)
    for part in ("train", "dev", "test"):
        d = {}
        for dom in ("CL", "IT"):
            p = find_file(f"{dom}_{part}.json"); assert p, f"{dom}_{part}.json not found"
            d.update(load_split(p, dom))
        out[part] = d
    for part in ("train", "dev"):
        for k, v in out[part].items():
            bad = [l for l in v["labels"] if l not in vocab]
            assert not bad, f"{k}: label(s) {sorted(set(bad))[:4]} not in label_vocab.json {vocab}"
    return out

class Enc:
    __slots__ = ("key", "id", "domain", "ids", "y", "soft")

    # It takes key, did (document id), domain (domain name), ids (token ids per sentence), y (labels) and soft (teacher probabilities) as input, stores a tokenized document and its labels and returns nothing.
    def __init__(self, key, did, domain, ids, y=None, soft=None):
        self.key, self.id, self.domain, self.ids, self.y, self.soft = key, did, domain, ids, y, soft

# It takes docs (documents), tok (tokenizer), label2i (label to id map) and max_sent_tok (max tokens per sentence) as input, tokenizes every sentence of every document and returns a dictionary of encoded documents.
def encode_docs(docs, tok, label2i, max_sent_tok):
    sep, out = tok.eos_id, {}
    for k, d in docs.items():
        ids = []
        for s in d["sents"]:
            t = list(tok.encode(s))[:max_sent_tok] or [sep]
            ids.append(t + [sep])
        y = np.array([label2i[l] for l in d["labels"]], dtype=np.int64) if d["labels"] is not None else None
        out[k] = Enc(k, d["id"], d["domain"], ids, y)
    return out

# It takes encs (encoded documents), soft_by_key (teacher probabilities per document) and keep_gold (keep true labels) as input, attaches the teacher probabilities to documents and returns a dictionary of documents with teacher probabilities.
def with_soft(encs, soft_by_key, keep_gold=False):
    out = {}
    for k, e in encs.items():
        if k not in soft_by_key: continue
        s = soft_by_key[k]; assert len(s) == len(e.ids), f"teacher probabilities for {k}: {len(s)} rows for {len(e.ids)} sentences"
        out[k] = Enc(e.key, e.id, e.domain, e.ids, e.y if keep_gold else None, s.astype(np.float32))
    return out

# It takes enc (encoded document), W (token limit), ctx (context size), rng (random generator) and step_frac (window step fraction) as input, packs sentences into windows that fit the token limit and returns a list of windows.
def windows(enc, W, ctx, rng=None, step_frac=0.5):
    n = len(enc.ids)
    if ctx == "sentence": return [(i, i + 1) for i in range(n)]
    L = [len(s) for s in enc.ids]; out, i = [], 0
    while i < n:
        j, tot = i, 0
        while j < n and (tot + L[j] <= W or j == i): tot += L[j]; j += 1
        out.append((i, j))
        if j >= n: break
        i += max(1, int((j - i) * (step_frac if rng is None else rng.uniform(0.4, 0.7))))
    return out

# It takes it (items) as input, counts the tokens in a window and returns a number of tokens.
def _tok_len(it):
    e, i, j = it
    return sum(len(e.ids[k]) for k in range(i, j))

# It takes items (list of items), pad_id (padding id), n_labels (number of labels) and device as input, builds padded tensors for one batch of windows and returns a dictionary of tensors.
def make_batch(items, pad_id, n_labels, device):
    B = len(items); S = max(j - i for _, i, j in items); T = max(_tok_len(it) for it in items)
    ids = np.full((B, T), pad_id, np.int64); st = np.zeros((B, S), np.int64); en = np.zeros((B, S), np.int64)
    sm = np.zeros((B, S), bool); y = np.full((B, S), -100, np.int64)
    soft = np.zeros((B, S, n_labels), np.float32) if any(e.soft is not None for e, _, _ in items) else None
    for b, (e, i, j) in enumerate(items):
        pos = 0
        for s, k in enumerate(range(i, j)):
            t = e.ids[k]; ids[b, pos:pos + len(t)] = t
            st[b, s], en[b, s], sm[b, s] = pos, pos + len(t) - 1, True
            pos += len(t)
            if e.y is not None: y[b, s] = e.y[k]
            if soft is not None and e.soft is not None: soft[b, s] = e.soft[k]
    out = dict(ids=torch.from_numpy(ids), st=torch.from_numpy(st), en=torch.from_numpy(en), sm=torch.from_numpy(sm), y=torch.from_numpy(y))
    if soft is not None: out["soft"] = torch.from_numpy(soft)
    return {k: v.to(device) for k, v in out.items()}

# It takes encs (encoded documents), cfg (settings), epoch_seed (seed for this epoch) and rare_set (rare labels) as input, groups training windows into batches of similar length and returns a list of batches.
def train_batches(encs, cfg, epoch_seed, rare_set=None):
    rng = random.Random(epoch_seed); items = []
    for e in encs.values():
        for (i, j) in windows(e, cfg["W"], cfg["ctx"], rng):
            items.append((e, i, j))
            if rare_set and cfg.get("rare_dup", 1) > 1 and e.y is not None and any(int(l) in rare_set for l in e.y[i:j]):
                items += [(e, i, j)] * (cfg["rare_dup"] - 1)
    rng.shuffle(items)
    bs, batches = cfg["bs"], []
    for g in range(0, len(items), bs * 50):
        grp = sorted(items[g:g + bs * 50], key=_tok_len)
        batches += [grp[k:k + bs] for k in range(0, len(grp), bs)]
    rng.shuffle(batches)
    return batches

class RRModel(nn.Module):

    # It takes bb (pre-trained backbone), n_labels (number of labels), ctx (context size), gru_hidden (GRU size), dropout (dropout rate) and pool (pooling type) as input, builds the sentence labelling model: backbone plus sentence layers and returns nothing.
    def __init__(self, bb, n_labels, ctx="window", gru_hidden=256, dropout=0.2, pool="meanlast"):
        super().__init__()
        self.bb_model, self.bb, self.pool, self.use_gru = bb.model, bb, pool, (ctx != "sentence")
        d = bb.d_model; din = 2 * d if pool == "meanlast" else d
        self.drop = nn.Dropout(dropout)
        if self.use_gru:
            self.gru = nn.GRU(din, gru_hidden, batch_first=True, bidirectional=True); dout = 2 * gru_hidden
            self.out = nn.Linear(dout, n_labels)
        else:
            self.mlp = nn.Sequential(nn.Linear(din, gru_hidden), nn.GELU(), nn.Dropout(dropout)); self.out = nn.Linear(gru_hidden, n_labels)

    # It takes ids (token ids), st (sentence start positions), en (sentence end positions) and sm (sentence mask) as input, runs the model and returns a label scores for each sentence.
    def forward(self, ids, st, en, sm):
        shim = copy.copy(self.bb); shim.model = self.bb_model
        h = shim.hidden(ids).float()
        d = h.size(-1)
        cs = F.pad(h.cumsum(1), (0, 0, 1, 0))
        gat = lambda idx: torch.gather(cs, 1, idx.unsqueeze(-1).expand(-1, -1, d))
        mean = (gat(en) - gat(st)) / (en - st).clamp(min=1).unsqueeze(-1).float()
        last = torch.gather(h, 1, en.unsqueeze(-1).expand(-1, -1, d))
        x = mean if self.pool == "mean" else last if self.pool == "last" else torch.cat([mean, last], -1)
        x = self.drop(x)
        if self.use_gru:
            lens = sm.sum(1).clamp(min=1).cpu()
            y, _ = self.gru(nn.utils.rnn.pack_padded_sequence(x, lens, batch_first=True, enforce_sorted=False))
            x, _ = nn.utils.rnn.pad_packed_sequence(y, batch_first=True, total_length=sm.size(1))
            return self.out(self.drop(x))
        return self.out(self.mlp(x))

# It takes model as input, counts trainable and total parameters and returns (trainable parameters, total parameters).
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad), sum(p.numel() for p in model.parameters())

# It takes model, head_lr (head learning rate), peft_lr (learning rate for the adapter weights) and wd (weight decay) as input, sets learning rates for the head and the other trainable weights and returns a list of parameter groups with learning rates.
def make_groups(model, head_lr, peft_lr, wd=0.01):
    head = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("bb_model.")]
    rest = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("bb_model.")]
    g = [dict(params=head, lr=head_lr, weight_decay=wd)]
    if rest: g.append(dict(params=rest, lr=peft_lr, weight_decay=wd))
    return g

# It takes gold (true labels), pred (predictions) and n_labels (number of labels) as input, computes macro F1 and other scores and returns a dictionary with macro F1 and other scores.
def scores(gold, pred, n_labels):
    cm = np.zeros((n_labels, n_labels), np.int64); np.add.at(cm, (gold, pred), 1)
    tp = np.diag(cm).astype(float); sup, pc = cm.sum(1), cm.sum(0)
    P = np.divide(tp, pc, out=np.zeros(n_labels), where=pc > 0); R = np.divide(tp, sup, out=np.zeros(n_labels), where=sup > 0)
    F1 = np.divide(2 * P * R, P + R, out=np.zeros(n_labels), where=(P + R) > 0)
    pres = (sup > 0) | (pc > 0)
    return dict(macro_f1=float(F1[pres].mean()), macro_p=float(P[pres].mean()), macro_r=float(R[pres].mean()), macro_f1_all13=float(F1.mean()),
                per_label=dict(P=P.tolist(), R=R.tolist(), F1=F1.tolist(), support=sup.tolist()), confusion=cm.tolist())

# It takes probs (probabilities), prior (label frequencies) and tau (rare-label boost) as input, boosts rare labels by adjusting the probabilities and returns a adjusted probabilities.
def adjust(probs, prior, tau):
    if tau == 0: return probs
    p = probs / np.power(prior[None, :], tau); return p / p.sum(1, keepdims=True)

# It takes encs (encoded documents) and n (number) as input, learns how likely each label follows another and returns a matrix of log transition probabilities.
def transitions(encs, n):
    T = np.ones((n, n))
    for e in encs.values():
        if e.y is not None and len(e.y) > 1: np.add.at(T, (e.y[:-1], e.y[1:]), 1)
    return np.log(T / T.sum(1, keepdims=True))

# It takes logp (log probabilities), logT (label transition scores) and lam (strength) as input, finds the best label sequence for the sentences and returns the best label id for each position.
def viterbi(logp, logT, lam):
    n, K = logp.shape
    if not lam or n < 2: return logp.argmax(1)
    dp, bp = logp[0].copy(), np.zeros((n, K), np.int64)
    for t in range(1, n):
        cand = dp[:, None] + lam * logT; bp[t] = cand.argmax(0); dp = cand.max(0) + logp[t]
    y = np.zeros(n, np.int64); y[-1] = int(dp.argmax())
    for t in range(n - 1, 0, -1): y[t - 1] = bp[t, y[t]]
    return y

# It takes p (probabilities), prior (label frequencies), tau (rare-label boost), logT (label transition scores) and lam (strength) as input, picks a label for every sentence and returns a label id for every sentence.
def decode(p, prior, tau, logT=None, lam=0.0):
    q = adjust(p, prior, tau)
    return q.argmax(1) if not lam else viterbi(np.log(q + 1e-9), logT, lam)

# It takes encs (encoded documents), probs (probabilities), n_labels (number of labels), prior (label frequencies), tau (rare-label boost), by_domain (also score per domain), logT (label transition scores) and lam (strength) as input, scores predictions on a split, also per domain and returns a dictionary of scores.
def score_split(encs, probs, n_labels, prior=None, tau=0.0, by_domain=True, logT=None, lam=0.0):
    # It takes keys as input, scores a group of documents and returns a dictionary of scores.
    def run(keys):
        g = np.concatenate([encs[k].y for k in keys]); p = np.concatenate([decode(probs[k], prior, tau, logT, lam) for k in keys]); return scores(g, p, n_labels)
    keys = [k for k in encs if encs[k].y is not None and k in probs]
    out = {"all": run(keys)}
    if by_domain:
        for dom in sorted({encs[k].domain for k in keys}):
            ks = [k for k in keys if encs[k].domain == dom]
            if ks: out[dom] = run(ks)
    return out

# It takes encs (encoded documents), n_labels (number of labels), power and cap (maximum value) as input, gives rare labels a higher loss weight and returns (weight per label, label frequencies).
def class_weights(encs, n_labels, power=0.5, cap=6.0):
    c = np.zeros(n_labels)
    for e in encs.values():
        if e.y is not None: c += np.bincount(e.y[e.y >= 0], minlength=n_labels)
    c = np.maximum(c, 1); w = (c.sum() / (n_labels * c)) ** power
    return torch.tensor(np.minimum(w / w.mean(), cap), dtype=torch.float32), c / c.sum()

# It takes e (error) as input, checks whether an error is a GPU out-of-memory error and returns True or False.
def _is_oom(e): return "out of memory" in str(e).lower()

# It takes model, encs (encoded documents), cfg (settings), device, n_labels (number of labels), pad_id (padding id) and bs (batch size) as input, predicts label probabilities for every sentence and returns probabilities for every document.
@torch.no_grad()
def predict_probs(model, encs, cfg, device, n_labels, pad_id, bs=None):
    model.eval(); bs = bs or cfg["bs"] * 2
    use_amp = device == "cuda"
    items = [(e, i, j) for e in encs.values() for (i, j) in windows(e, cfg["W"], cfg["ctx"], None)]
    order = sorted(range(len(items)), key=lambda k: _tok_len(items[k]))
    acc = {k: np.zeros((len(e.ids), n_labels), np.float32) for k, e in encs.items()}; wsum = {k: np.zeros(len(e.ids), np.float32) for k, e in encs.items()}
    dmodel = nn.DataParallel(model) if (device == "cuda" and torch.cuda.device_count() > 1) else model
    p = 0
    while p < len(order):
        part = [items[k] for k in order[p:p + bs]]
        b = make_batch(part, pad_id, n_labels, device)
        try:
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                pr = torch.softmax(dmodel(b["ids"], b["st"], b["en"], b["sm"]).float(), -1).cpu().numpy()
        except RuntimeError as ex:
            if _is_oom(ex) and bs > 1: torch.cuda.empty_cache(); bs = max(1, bs // 2); continue
            raise
        for r, (e, i, j) in enumerate(part):
            for s, k in enumerate(range(i, j)):
                pos = (s + 0.5) / (j - i); w = 1.0 - 0.8 * abs(2 * pos - 1)
                acc[e.key][k] += w * pr[r, s]; wsum[e.key][k] += w
        p += len(part)
    return {k: acc[k] / np.maximum(wsum[k], 1e-9)[:, None] for k in acc}

PROGRESS = dict(total=0.0, done=0.0, skipped=0.0, t0=None)

# It takes sec (seconds) as input, formats seconds as hours:minutes:seconds and returns time as text.
def _hms(sec):
    sec = int(max(0, sec)); return f"{sec // 3600}h{(sec % 3600) // 60:02d}m"

# It shows overall progress and time left and returns progress text.
def progress_line():
    P = PROGRESS
    if not P["total"] or not P["t0"] or P["done"] <= 0: return ""
    el = time.time() - P["t0"]; rate = P["done"] / max(1e-9, el); left = max(0.0, P["total"] - P["done"] - P["skipped"])
    return f"[overall {100 * min(1.0, (P['done'] + P['skipped']) / P['total']):.0f}% | elapsed {_hms(el)} | ETA {_hms(left / rate)}]"

# It takes model as input, copies the trainable weights and returns a dictionary of trainable weights.
def _tstate(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in names}

# It takes model, train_encs (encoded training documents), val_encs (encoded validation documents), cfg (settings), groups (parameter groups with learning rates), device, n_labels (number of labels), pad_id (padding id), log (print function) and ckpt_path (checkpoint path) as input, trains the model with warm-up, cosine learning rate and validation checks and returns training history and the best weights.
def fit(model, train_encs, val_encs, cfg, groups, device, n_labels, pad_id, log=print, ckpt_path=None):
    use_amp = device == "cuda"
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98), weight_decay=cfg.get("wd", 0.01)); base_lr = [g["lr"] for g in groups]
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    cw, prior = class_weights(train_encs, n_labels, cfg.get("class_power", 0.5)); cw = cw.to(device)
    rare = {i for i, p in enumerate(prior) if p < cfg.get("rare_share", 0.03)}
    dmodel = nn.DataParallel(model) if (device == "cuda" and torch.cuda.device_count() > 1) else model
    accum, E = cfg["accum"], cfg["epochs"]
    n_b = len(train_batches(train_encs, cfg, 0, rare)); steps_per_epoch = max(1, n_b // accum); total = steps_per_epoch * E
    warm = max(1, int(cfg.get("warmup", 0.06) * total))
    log(f"  micro-batches/epoch {n_b} (bs {cfg['bs']} x accum {accum}), optimizer steps/epoch {steps_per_epoch}, total {total}; rare labels {sorted(rare)}")
    kd_a, kd_T = cfg.get("kd_alpha", 0.0), cfg.get("kd_T", 2.0)
    train_params = [p for g in groups for p in g["params"]]
    step, ep0, best, best_ep, hist, best_state, avg_state, avg_n = 0, 0, -1.0, -1, [], None, None, 0
    snaps, snap_eps = {}, set(cfg.get("snap_epochs") or [])
    if ckpt_path and os.path.exists(ckpt_path):
        try:
            ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
            model.load_state_dict(ck["state"], strict=False); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
            step, ep0, best, best_ep, hist = ck["step"], ck["ep"], ck["best"], ck["best_ep"], ck["hist"]
            best_state, avg_state, avg_n = ck["best_state"], ck["avg_state"], ck["avg_n"]; snaps = ck.get("snaps", {})
            PROGRESS["skipped"] += ep0 * len(train_encs); log(f"  resumed from {os.path.basename(ckpt_path)} at epoch {ep0 + 1}/{E}"); del ck
        except Exception as ex:
            log(f"  could not resume ({str(ex)[:100]}) -> training from scratch"); step, ep0, best, best_ep, hist, best_state, avg_state, avg_n = 0, 0, -1.0, -1, [], None, None, 0
    t_save, bs_now = time.time(), cfg["bs"]
    for ep in range(ep0, E):
        model.train(); t0 = time.time(); run, nrun = torch.zeros((), device=device), 0
        if PROGRESS["t0"] is None: PROGRESS["t0"] = t0
        batches = train_batches(train_encs, cfg, cfg.get("seed", 0) * 1000 + ep, rare)
        ep_steps = max(1, len(batches) // accum)
        try:
            from tqdm.auto import tqdm as _tqdm
            bar = _tqdm(total=ep_steps, desc=f"epoch {ep + 1}/{E}", mininterval=20, leave=False, dynamic_ncols=True)
        except Exception:
            bar = None
        for s0 in range(0, ep_steps * accum, accum):
            lr_f = (step + 1) / warm if step < warm else max(0.02, 0.5 * (1 + math.cos(math.pi * min(1.0, (step - warm) / max(1, total - warm)))))
            for g, b0 in zip(opt.param_groups, base_lr): g["lr"] = b0 * lr_f
            if not batches[s0:s0 + accum]: break
            for part in batches[s0:s0 + accum]:
                b = make_batch(part, pad_id, n_labels, device)
                try:
                    with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                        lg = dmodel(b["ids"], b["st"], b["en"], b["sm"]).float()
                    ym = b["y"] != -100
                    ce = F.cross_entropy(lg[ym], b["y"][ym], weight=cw, label_smoothing=cfg.get("label_smooth", 0.0)) if bool(ym.any()) else lg.sum() * 0.0
                    loss = ce
                    if kd_a > 0 and "soft" in b:
                        has = (b["soft"].sum(-1) > 0) & b["sm"]
                        if bool(has.any()):
                            tgt = b["soft"][has].clamp_min(1e-9) ** (1 / kd_T); tgt = tgt / tgt.sum(-1, keepdim=True)
                            kd = F.kl_div(F.log_softmax(lg[has] / kd_T, -1), tgt, reduction="batchmean") * kd_T * kd_T
                            frac = has.float().sum() / (ym | has).float().sum().clamp(min=1)
                            loss = (1 - kd_a * frac) * ce + kd_a * frac * kd
                    scaler.scale(loss / accum).backward(); run += ce.detach().float(); nrun += 1
                except RuntimeError as ex:
                    if _is_oom(ex):
                        raise RuntimeError("CUDA out of memory: lower cfg['bs'] (windows per micro-batch) or cfg['W'], and raise cfg['accum'] to keep the same effective batch") from ex
                    raise
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(train_params, cfg.get("clip", 1.0))
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
            if bar is not None:
                bar.update(1)
                if step % 25 == 0: bar.set_postfix(ce=f"{float(run) / max(1, nrun):.3f}")
        if bar is not None: bar.close()
        PROGRESS["done"] += len(train_encs)
        rec = dict(epoch=ep + 1, train_ce=float(run) / max(1, nrun), secs=time.time() - t0)
        if val_encs:
            pr = predict_probs(model, val_encs, cfg, device, n_labels, pad_id); sc = score_split(val_encs, pr, n_labels)
            rec.update(val_macro_F1=sc["all"]["macro_f1"], **{f"val_{d}": sc[d]["macro_f1"] for d in sc if d != "all"})
            if rec["val_macro_F1"] > best: best, best_ep, best_state = rec["val_macro_F1"], ep + 1, _tstate(model)
        if cfg.get("swa_last", 0) and ep + 1 > E - cfg["swa_last"]:
            cur = _tstate(model)
            if avg_state is None: avg_state, avg_n = {k: v.float() for k, v in cur.items()}, 1
            else:
                avg_n += 1
                for k, v in cur.items(): avg_state[k] += (v.float() - avg_state[k]) / avg_n
        if ep + 1 in snap_eps: snaps[ep + 1] = _tstate(model)
        hist.append(rec)
        log("  " + "  ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()) +
            f"  | epoch {ep + 1}/{E}, run ETA {_hms((time.time() - t0) * (E - ep - 1))}  " + progress_line())
        if ckpt_path and ep + 1 < E and time.time() - t_save >= cfg.get("save_every", 1800):
            torch.save(dict(state=_tstate(model), opt=opt.state_dict(), scaler=scaler.state_dict(), step=step, ep=ep + 1, best=best, best_ep=best_ep,
                            hist=hist, best_state=best_state, avg_state=avg_state, avg_n=avg_n, snaps=snaps), ckpt_path + ".tmp")
            os.replace(ckpt_path + ".tmp", ckpt_path); t_save = time.time(); log(f"  [saved resumable checkpoint after epoch {ep + 1}]")
    last_state = _tstate(model)
    if best_state is None: best_state, best_ep = last_state, len(hist)
    if avg_state is not None: avg_state = {k: v.to(last_state[k].dtype) if last_state[k].is_floating_point() else last_state[k] for k, v in avg_state.items()}
    if ckpt_path and os.path.exists(ckpt_path): os.remove(ckpt_path)
    return hist, best_ep, best_state, last_state, avg_state, snaps

# It takes model and state (weights) as input, loads saved weights into the model and returns nothing.
def load_state(model, state):
    miss = model.load_state_dict(state, strict=False); assert not miss.unexpected_keys, miss.unexpected_keys

# It takes s (random seed) as input, sets random seeds so runs are repeatable and returns nothing.
def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

# It takes path (file path) and probs (probabilities) as input, saves predicted probabilities to a file and returns nothing.
def save_probs(path, probs):
    keys = sorted(probs); np.savez_compressed(path, keys=np.array(keys), **{f"p{i}": probs[k].astype(np.float16) for i, k in enumerate(keys)})

# It takes path (file path) as input, loads saved probabilities from a file and returns probabilities per document.
def load_probs(path):
    z = np.load(path, allow_pickle=True); return {str(k): z[f"p{i}"].astype(np.float32) for i, k in enumerate(z["keys"])}

# It takes plist (list of probabilities) as input, averages several probability dictionaries and returns averaged probabilities.
def avg_probs(plist):
    return {k: sum(p[k] for p in plist) / len(plist) for k in plist[0]}

# It takes test_encs (encoded test documents), probs (probabilities), labels (label names), prior (label frequencies), tau (rare-label boost), logT (label transition scores) and lam (strength) as input, makes the label list for every test document and returns a dictionary from document id to labels.
def predictions_for(test_encs, probs, labels, prior, tau, logT=None, lam=0.0):
    pred = {}
    for k, e in test_encs.items():
        assert e.id not in pred, (f"document id {e.id!r} occurs in BOTH CL and IT test files - one prediction.json cannot hold both. "
                                  "Look at the evaluation script of the course to see how the two domains must be combined, then change this function.")
        lab = decode(probs[k], prior, tau, logT, lam); pred[e.id] = [labels[int(i)] for i in lab]
    return pred

# It takes pred (predictions), test_docs (test documents) and labels (label names) as input, checks the predictions match the test documents and returns True (stops with an error if something is wrong).
def check_prediction(pred, test_docs, labels):
    ids_t = {d["id"] for d in test_docs.values()}
    assert ids_t == set(pred), f"id mismatch: missing {sorted(ids_t - set(pred))[:3]} unknown {sorted(set(pred) - ids_t)[:3]}"
    lens = {d["id"]: len(d["sents"]) for d in test_docs.values()}; vs = set(labels)
    for k, v in pred.items():
        assert len(v) == lens[k], f"{k}: {len(v)} labels for {lens[k]} sentences"
        assert all(l in vs for l in v), f"{k}: label not in label_vocab"
    return True

# It takes pred (predictions) and out_zip (zip path) as input, writes predictions into the submission zip and returns a path of the zip.
def write_submission(pred, out_zip):
    with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z: z.writestr("prediction.json", json.dumps(pred, ensure_ascii=False))
    with zipfile.ZipFile(out_zip) as z: assert z.namelist() == ["prediction.json"]
    return out_zip

# It takes cm (confusion matrix), labels (label names), path (file path) and title as input, plots the confusion matrix and returns nothing.
def plot_confusion(cm, labels, path, title=""):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    cm = np.array(cm, float); R = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(9, 8)); im = ax.imshow(R, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("predicted"); ax.set_ylabel("gold (row-normalised)"); ax.set_title(title); fig.colorbar(im); fig.tight_layout(); fig.savefig(path, dpi=130); plt.close(fig)

# It takes name, build (function that builds the model), cfg (settings), data (dataset splits), out_dir (output folder), work_dir (work folder), resume_dirs (folders of earlier runs), log (print function), soft_train (teacher-labelled training documents), taus (rare-label boost values), dev_seeds (seeds for the dev stage), full_seeds (seeds for the full stage) and stages (stages to run) as input, runs one full experiment: train, evaluate and save outputs and returns a results summary.
def run_experiment(name, build, cfg, data, out_dir, work_dir, resume_dirs=(), log=print, soft_train=None, taus=(0.0, 0.5, 1.0),
                   dev_seeds=(0,), full_seeds=(0,), stages=("dev", "full")):
    os.makedirs(out_dir, exist_ok=True); os.makedirs(work_dir, exist_ok=True)
    labels, K, pad_id = data["labels"], len(data["labels"]), data["pad_id"]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    _, prior = class_weights({**data["train"], **data["dev"]}, K)
    summary = dict(name=name, labels=labels, cfg={k: v for k, v in cfg.items() if isinstance(v, (int, float, str, bool, list, tuple, type(None)))}, stages={})
    n_docs = {"dev": len(data["train"]), "full": len(data["train"]) + len(data["dev"])}
    PROGRESS.update(total=cfg["epochs"] * (("dev" in stages) * n_docs["dev"] * len(dev_seeds) + ("full" in stages) * n_docs["full"] * len(full_seeds)), done=0.0, skipped=0.0, t0=None)

    # It takes fn (file name) as input, finds a saved file from this or an earlier run and returns a file path (or None).
    def have(fn):
        for d in (work_dir,) + tuple(resume_dirs):
            p = os.path.join(d, fn)
            if os.path.exists(p): return p
        return None

    lams = list(cfg.get("vit_lams") or [0.0]); vs = lambda l: f"_vit{l:g}" if l else ""
    # It takes stage (stage name), kind (type), probs_test (test probabilities) and logT (label transition scores) as input, writes submission files for the given predictions and returns nothing.
    def emit(stage, kind, probs_test, logT):
        for t in taus:
            for l in lams:
                pred = predictions_for(data["test"], probs_test, labels, prior, t, logT, l); check_prediction(pred, data["raw"]["test"], labels)
                fn = f"{name}__{stage}_{kind}__tau{t:g}{vs(l)}_submission.zip"; write_submission(pred, os.path.join(out_dir, fn)); log(f"    wrote {fn}")

    for stage in [s for s in ("dev", "full") if s in stages]:
        seeds = dev_seeds if stage == "dev" else full_seeds
        log(f"\n===== {name}: stage '{stage}' ({'train -> dev' if stage == 'dev' else 'train + dev, all documents'}), seeds {list(seeds)} =====")
        from collections import defaultdict
        per_kind, per_kind_dev, hist_all = defaultdict(list), defaultdict(list), []
        logT = transitions({**data["train"], **(data["dev"] if stage == "full" else {})}, K)
        for seed in seeds:
            fn = f"{name}_{stage}_s{seed}.npz"; got = have(fn)
            if got:
                z = np.load(got, allow_pickle=True); log(f"  seed {seed}: found finished run {fn} -> skipped"); PROGRESS["skipped"] += cfg["epochs"] * n_docs[stage]
                for kind in [f[:-10] for f in z.files if f.endswith("_test_keys")]:
                    if True:
                        per_kind[kind].append({str(k): z[f"{kind}_test_{i}"].astype(np.float32) for i, k in enumerate(z[f"{kind}_test_keys"])})
                        if stage == "dev": per_kind_dev[kind].append({str(k): z[f"{kind}_dev_{i}"].astype(np.float32) for i, k in enumerate(z[f"{kind}_dev_keys"])})
                hist_all.append(json.loads(str(z["hist"]))); continue
            seed_all(seed); model, groups, info = build(seed); model.to(device)
            tr, al = count_params(model); log(f"  seed {seed}: trainable parameters {tr:,} of {al:,} ({100 * tr / al:.2f} %)  {info or ''}")
            train = dict(data["train"]);
            if stage == "full": train.update(data["dev"])
            if soft_train: train.update(with_soft(train, soft_train[stage], keep_gold=True))
            c = dict(cfg, seed=seed)
            hist, best_ep, bst, lst, ast, snaps = fit(model, train, data["dev"] if stage == "dev" else None, c, groups, device, K, pad_id, log, os.path.join(work_dir, f"{name}_{stage}_s{seed}.resume.pt"))
            store, kinds = {}, ([("best", bst), ("final", lst), ("avg", ast)] if stage == "dev" else [("final", lst), ("avg", ast)])
            kinds += [(f"ep{e}", st) for e, st in sorted(snaps.items()) if e != cfg["epochs"]]
            kinds = [kv for kv in kinds if kv[1] is not None]
            for kind, st in kinds:
                if st is None: continue
                load_state(model, st)
                pt = predict_probs(model, data["test"], c, device, K, pad_id); per_kind[kind].append(pt)
                store[f"{kind}_test_keys"] = np.array(sorted(pt)); store.update({f"{kind}_test_{i}": pt[k].astype(np.float16) for i, k in enumerate(sorted(pt))})
                if stage == "dev":
                    pd_ = predict_probs(model, data["dev"], c, device, K, pad_id); per_kind_dev[kind].append(pd_)
                    store[f"{kind}_dev_keys"] = np.array(sorted(pd_)); store.update({f"{kind}_dev_{i}": pd_[k].astype(np.float16) for i, k in enumerate(sorted(pd_))})
            if cfg.get("dump_train") and lst is not None:
                load_state(model, lst); ptr = predict_probs(model, train, c, device, K, pad_id); save_probs(os.path.join(out_dir, f"{name}_{stage}_s{seed}_train_soft.npz"), ptr)
            np.savez_compressed(os.path.join(work_dir, fn), hist=json.dumps(hist), best_ep=best_ep, **store); hist_all.append(hist)
            del model
            if device == "cuda": torch.cuda.empty_cache()
        info_stage = dict(history=hist_all, scores={})
        for kind in sorted(per_kind, key=lambda k: (k not in ("best", "final", "avg"), k)):
            if not per_kind[kind]: continue
            pt = avg_probs(per_kind[kind]); save_probs(os.path.join(out_dir, f"{name}_{stage}_{kind}_test_probs.npz"), pt)
            if stage == "dev":
                pdv = avg_probs(per_kind_dev[kind]); save_probs(os.path.join(out_dir, f"{name}_dev_{kind}_dev_probs.npz"), pdv)
                for t in taus:
                  for l in lams:
                    sc = score_split(data["dev"], pdv, K, prior, t, logT=logT, lam=l); info_stage["scores"][f"{kind}_tau{t:g}{vs(l)}"] = {d: {k: v for k, v in s.items()} for d, s in sc.items()}
                    log(f"  dev Macro-F1 [{kind:5s} tau {t:g}{' viterbi ' + format(l, 'g') if l else ''}]  all {sc['all']['macro_f1']:.4f}  " + "  ".join(f"{d} {sc[d]['macro_f1']:.4f}" for d in sc if d != 'all') + f"  (all 13 labels: {sc['all']['macro_f1_all13']:.4f})")
                sc0 = info_stage["scores"][f"{kind}_tau{taus[0]:g}"]["all"]
                log("    per label (first tau, no viterbi):  " + "  ".join(f"{l} F1 {f:.2f} (n={int(n)})" for l, f, n in zip(labels, sc0["per_label"]["F1"], sc0["per_label"]["support"])))
                plot_confusion(sc0["confusion"], labels, os.path.join(out_dir, f"{name}_dev_{kind}_confusion.png"), f"{name} dev {kind}")
            emit(stage, kind, pt, logT)
        summary["stages"][stage] = info_stage
        json.dump(summary, open(os.path.join(out_dir, f"{name}_summary.json"), "w"), indent=1)
    return summary

Overwriting rr_core.py


In [10]:
%%writefile rr_peft.py
# Fine-tuning methods: LoRA, LoRMA, adapters (saved as rr_peft.py)
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# It takes model and name as input, gets a sub-layer of the model by its name and returns the sub-layer.
def _get(model, name):
    m = model
    for p in name.split("."):
        m = getattr(m, p)
    return m

# It takes model, name and new (new layer) as input, replaces a sub-layer of the model by its name and returns nothing.
def _set(model, name, new):
    parts = name.split(".")
    parent = _get(model, ".".join(parts[:-1])) if len(parts) > 1 else model
    setattr(parent, parts[-1], new)

class LoRALinear(nn.Module):
    # It takes base (base linear layer), r (rank), alpha (scale of the low-rank update), dropout (dropout rate) and rs (use rsLoRA scaling) as input, creates the small LoRA matrices around a linear layer and returns nothing.
    def __init__(self, base: nn.Linear, r=8, alpha=16, dropout=0.05, rs=False):
        super().__init__()
        self.base, self.s = base, (alpha / math.sqrt(r) if rs else alpha / r)
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(dropout)

    # It takes x (input tensor) as input, runs the linear layer plus the LoRA update and returns the output of the updated layer.
    def forward(self, x):
        return self.base(x) + self.s * F.linear(F.linear(self.drop(x), self.A.to(x.dtype)), self.B.to(x.dtype))

class LoRMALinear(nn.Module):

    # It takes base (base linear layer), r (rank) and alpha (scale of the low-rank update) as input, creates the small LoRMA matrices around a linear layer and returns nothing.
    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base, self.s = base, alpha / r
        o = base.out_features
        self.A = nn.Parameter(torch.empty(r, o)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(o, r))

    # It takes x (input tensor) as input, runs the linear layer with the LoRMA update and returns the output of the updated layer.
    def forward(self, x):
        y0 = F.linear(x, self.base.weight.to(x.dtype))
        y = y0 + self.s * F.linear(F.linear(y0, self.A.to(x.dtype)), self.B.to(x.dtype))
        return y if self.base.bias is None else y + self.base.bias.to(x.dtype)

# It takes model as input, freezes all weights of the model and returns nothing.
def freeze(model):
    for p in model.parameters():
        p.requires_grad_(False)

# It takes bb (pre-trained backbone), kind (type), names (layer names), r (rank), alpha (scale of the low-rank update) and dropout (dropout rate) as input, freezes the model and wraps chosen linear layers with LoRA or LoRMA and returns a number of layers wrapped.
def apply_linear_wrappers(bb, kind, names, r=8, alpha=16, dropout=0.05):
    freeze(bb.model)
    n = 0
    for name in names:
        base = _get(bb.model, name)
        if not isinstance(base, nn.Linear):
            continue
        new = LoRMALinear(base, r, alpha) if kind == "lorma" else LoRALinear(base, r, alpha, dropout, rs=(kind == "rslora"))
        _set(bb.model, name, new); n += 1
    assert n > 0, "no Linear layers wrapped - check the name list"
    return n

class Adapter(nn.Module):
    # It takes d (model size), bottleneck (adapter size) and dropout (dropout rate) as input, creates a small adapter that starts as no change and returns nothing.
    def __init__(self, d, bottleneck=64, dropout=0.1):
        super().__init__()
        self.down, self.up, self.drop = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d), nn.Dropout(dropout)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    # It takes h (hidden vectors) as input, runs the block and adds the adapter output and returns output with the adapter added.
    def forward(self, h):
        return h + self.up(self.drop(F.gelu(self.down(h))))

# It takes bb (pre-trained backbone) as input, returns the list of model blocks.
def get_blocks(bb):
    if hasattr(bb, "block_modules"):
        return list(bb.block_modules())
    return list(bb.model.blocks)

# It takes bb (pre-trained backbone), bottleneck (adapter size), dropout (dropout rate) and every (how often) as input, freezes the model and adds an adapter after the blocks and returns a number of adapters added.
def add_block_adapters(bb, bottleneck=64, dropout=0.1, every=1):
    freeze(bb.model)
    blocks = get_blocks(bb)
    n = 0
    for i, blk in enumerate(blocks):
        if i % every: continue
        blk.add_module("ner_adapter", Adapter(bb.d_model, bottleneck, dropout))

        # It takes mod (module), inp (layer input) and out (block output) as input, adds the adapter output to the block output and returns block output plus adapter output.
        def hook(mod, inp, out):
            ad = mod.ner_adapter
            if isinstance(out, tuple):
                return (ad(out[0]),) + tuple(out[1:])
            return ad(out)
        blk.register_forward_hook(hook); n += 1
    return n

# It takes bb (pre-trained backbone), last_k_blocks (number of top blocks to train), train_norms (also train norm layers) and train_embeddings (also train embeddings) as input, freezes the model except the last few blocks and returns a number of trainable parameters.
def partial_finetune(bb, last_k_blocks=4, train_norms=True, train_embeddings=False):
    freeze(bb.model)
    blocks = get_blocks(bb)
    for blk in blocks[-last_k_blocks:]:
        for p in blk.parameters(): p.requires_grad_(True)
    if train_norms:
        for n, p in bb.model.named_parameters():
            if "norm_f" in n or n.startswith("norm.") or n.startswith("ln_f"):
                p.requires_grad_(True)
    if train_embeddings:
        for n, p in bb.model.named_parameters():
            if "embed" in n or "tok" in n.split(".")[0]: p.requires_grad_(True)
    return sum(p.numel() for p in bb.model.parameters() if p.requires_grad)

# It takes bb (pre-trained backbone), lr_top (top-layer learning rate), decay (decay factor), head_params (head weights), head_lr (head learning rate) and wd (weight decay) as input, gives lower layers a smaller learning rate than top layers and returns a list of parameter groups with learning rates.
def layerwise_lr_groups(bb, lr_top, decay=0.85, head_params=(), head_lr=1e-3, wd=0.01):
    blocks = get_blocks(bb)
    n = len(blocks)
    seen, groups = set(), []
    for i, blk in enumerate(blocks):
        ps = [p for p in blk.parameters() if p.requires_grad]
        for p in ps: seen.add(id(p))
        groups.append(dict(params=ps, lr=lr_top * decay ** (n - 1 - i), weight_decay=wd))
    rest = [p for p in bb.model.parameters() if p.requires_grad and id(p) not in seen]
    groups.append(dict(params=rest, lr=lr_top * decay ** n, weight_decay=wd))
    groups.append(dict(params=list(head_params), lr=head_lr, weight_decay=wd))
    return [g for g in groups if g["params"]]

Overwriting rr_peft.py


In [11]:
%%writefile arch_graphprop.py
# GraphProp model code (saved as arch_graphprop.py)
import os, json, zipfile, tempfile, shutil, re as _re
from collections import Counter, defaultdict as _defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

ARCH = "graphprop"

class TokenPosEmbed(nn.Module):
    # It takes vocab_size (vocabulary size), dim (model size) and max_len (max length) as input, creates the token and position embedding tables and returns nothing.
    def __init__(self, vocab_size, dim, max_len=1024):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, dim)
        self.pos = nn.Embedding(max_len, dim)

    # It takes input_ids (token ids) and pos_offset (position offset) as input, adds token and position embeddings for the input and returns embedding vectors.
    def forward(self, input_ids, pos_offset=0):
        B, T = input_ids.shape
        positions = (torch.arange(T, device=input_ids.device) + pos_offset).unsqueeze(0).expand(B, -1)
        return self.tok(input_ids) + self.pos(positions)

class FeedForward(nn.Module):
    # It takes dim (model size), mult (size multiplier) and dropout (dropout rate) as input, creates the feed-forward layer and returns nothing.
    def __init__(self, dim, mult=4, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.net = nn.Sequential(
            nn.Linear(dim, dim * mult),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * mult, dim),
            nn.Dropout(dropout),
        )

    # It takes x (input tensor) as input, runs the feed-forward layer and returns output vectors.
    def forward(self, x):
        return x + self.net(self.norm(x))

class GraphPropLayer(nn.Module):
    # It takes dim (model size), window (window size) and edge_dim (edge size) as input, creates the graph message-passing layer and returns nothing.
    def __init__(self, dim, window=16, edge_dim=64):
        super().__init__()
        self.window = window
        self.norm = nn.LayerNorm(dim)
        self.edge_key = nn.Linear(dim, edge_dim)
        self.edge_mlp = nn.Sequential(nn.Linear(edge_dim * 2, edge_dim), nn.GELU(), nn.Linear(edge_dim, 1))
        self.value_proj = nn.Linear(dim, dim)
        self.rel_emb = nn.Embedding(window, dim)
        self.out_proj = nn.Linear(dim, dim)

    # It takes x (input tensor) as input, passes messages from nearby previous tokens to each token and returns output vectors.
    def forward(self, x):
        B, T, D = x.shape
        W = self.window
        h = self.norm(x)

        K = self.edge_key(h)
        e = K.size(-1)
        pad_rows = self.edge_key.bias.to(K.dtype).expand(B, W - 1, -1)
        K_all = torch.cat([pad_rows, K], dim=1)
        w1, b1 = self.edge_mlp[0].weight, self.edge_mlp[0].bias
        A = F.linear(K, w1[:, :e], b1)
        Bn = F.linear(K_all, w1[:, e:])
        z = A.unsqueeze(2) + Bn.unfold(1, W, 1).permute(0, 1, 3, 2)
        edge_w = torch.sigmoid(self.edge_mlp[2](self.edge_mlp[1](z))).squeeze(-1)

        wv, bv = self.value_proj.weight, self.value_proj.bias
        V = F.pad(F.linear(h, wv), (0, 0, W - 1, 0))
        R = F.linear(self.rel_emb.weight, wv, bv)
        t_idx = torch.arange(T, device=x.device).unsqueeze(1) + torch.arange(W, device=x.device).unsqueeze(0)
        M = torch.zeros(B, T, T + W - 1, dtype=edge_w.dtype, device=x.device)
        M.scatter_(2, t_idx.unsqueeze(0).expand(B, -1, -1), edge_w)
        num = torch.bmm(M, V.to(M.dtype)) + torch.matmul(edge_w, R.to(edge_w.dtype))

        denom = edge_w.sum(-1, keepdim=True).clamp_min(1e-6)
        agg = num / denom
        out = self.out_proj(agg)
        return x + out

    # It takes h (normalised hidden vectors), valid (mask of valid neighbours) and drop_self (skip the token itself) as input, computes the message sums used by the layer and returns (message sum, edge weight sum).
    def _num_den(self, h, valid=None, drop_self=False):
        B, T, D = h.shape
        W = self.window
        K = self.edge_key(h)
        e = K.size(-1)
        pad_rows = self.edge_key.bias.to(K.dtype).expand(B, W - 1, -1)
        K_all = torch.cat([pad_rows, K], dim=1)
        w1, b1 = self.edge_mlp[0].weight, self.edge_mlp[0].bias
        A = F.linear(K, w1[:, :e], b1)
        Bn = F.linear(K_all, w1[:, e:])
        z = A.unsqueeze(2) + Bn.unfold(1, W, 1).permute(0, 1, 3, 2)
        edge_w = torch.sigmoid(self.edge_mlp[2](self.edge_mlp[1](z))).squeeze(-1)
        if valid is not None:
            vpad = torch.cat([torch.ones(B, W - 1, dtype=edge_w.dtype, device=h.device),
                              valid.to(edge_w.dtype)], dim=1)
            edge_w = edge_w * vpad.unfold(1, W, 1)
        if drop_self:
            edge_w = torch.cat([edge_w[..., :W - 1], torch.zeros_like(edge_w[..., :1])], dim=-1)
        wv, bv = self.value_proj.weight, self.value_proj.bias
        V = F.pad(F.linear(h, wv), (0, 0, W - 1, 0))
        R = F.linear(self.rel_emb.weight, wv, bv)
        t_idx = torch.arange(T, device=h.device).unsqueeze(1) + torch.arange(W, device=h.device).unsqueeze(0)
        M = torch.zeros(B, T, T + W - 1, dtype=edge_w.dtype, device=h.device)
        M.scatter_(2, t_idx.unsqueeze(0).expand(B, -1, -1), edge_w)
        num = torch.bmm(M, V.to(M.dtype)) + torch.matmul(edge_w, R.to(edge_w.dtype))
        return num, edge_w.sum(-1, keepdim=True)

    # It takes x (input tensor) and valid (mask of valid neighbours) as input, passes messages from tokens on both sides and returns output vectors.
    def forward_bidirectional(self, x, valid=None):
        h = self.norm(x)
        nc, dc = self._num_den(h, valid)
        vr = valid.flip(1) if valid is not None else None
        nr, dr = self._num_den(h.flip(1), vr, drop_self=True)
        num = nc + nr.flip(1)
        den = (dc + dr.flip(1)).clamp_min(1e-6)
        return x + self.out_proj(num / den)

class LegalGraphPropLM(nn.Module):
    # It takes vocab_size (vocabulary size), dim (model size), n_layers (number of layers), window (window size), edge_dim (edge size), max_len (max length), dropout (dropout rate) and tie (tie embeddings) as input, builds the full GraphProp language model and returns nothing.
    def __init__(self, vocab_size, dim=512, n_layers=6, window=16, edge_dim=64, max_len=1024,
                 dropout=0.1, tie=True):
        super().__init__()
        self.embed = TokenPosEmbed(vocab_size, dim, max_len)
        self.drop = nn.Dropout(dropout)
        layers = []
        for _ in range(n_layers):
            layers.append(GraphPropLayer(dim, window, edge_dim))
            layers.append(FeedForward(dim, dropout=dropout))
        self.blocks = nn.ModuleList(layers)
        self.norm_f = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        if tie:
            self.lm_head.weight = self.embed.tok.weight

        self.apply(self._init_weights)
        residual_scale = (2 * n_layers) ** -0.5
        for blk in self.blocks:
            if isinstance(blk, GraphPropLayer):
                nn.init.normal_(blk.out_proj.weight, mean=0.0, std=0.02 * residual_scale)
            elif isinstance(blk, FeedForward):
                nn.init.normal_(blk.net[-2].weight, mean=0.0, std=0.02 * residual_scale)

    # It sets the starting values of the weights and returns nothing.
    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    # It takes input_ids (token ids), pos_offset (position offset), targets (target token ids), last_idx (last token positions), pad_id (padding id) and label_smoothing (label smoothing) as input, runs the model and returns next-token scores.
    def forward(self, input_ids, pos_offset=0, targets=None, last_idx=None, pad_id=None, label_smoothing=0.0):
        x = self.drop(self.embed(input_ids, pos_offset=pos_offset))
        for blk in self.blocks:
            x = blk(x)
        h = self.norm_f(x)
        if last_idx is not None:
            rows = torch.arange(h.size(0), device=h.device)
            return self.lm_head(h[rows, last_idx])
        logits = self.lm_head(h)
        if targets is None:
            return logits
        loss_sum = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1),
                                   ignore_index=pad_id, reduction="sum", label_smoothing=label_smoothing)
        n_tok = (targets != pad_id).sum()
        return loss_sum.reshape(1), n_tok.reshape(1)

# It maps every byte to a printable character for the byte-level tokenizer and returns a dictionary from byte value to character.
def _bytes_to_unicode():
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("\xa1"), ord("\xac") + 1)) + \
         list(range(ord("\xae"), ord("\xff") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, [chr(c) for c in cs]))

_BYTE_ENCODER = _bytes_to_unicode()
_BYTE_DECODER = {v: k for k, v in _BYTE_ENCODER.items()}
_PRETOK_RE = _re.compile(r"\'s|\'t|\'re|\'ve|\'m|\'ll|\'d| ?[A-Za-z]+| ?[0-9]+| ?[^\sA-Za-z0-9]+|\s+")

class LegalByteBPETokenizer:

    PAD_TOKEN = "<pad>"
    EOS_TOKEN = "<|endoftext|>"

    # It creates an empty tokenizer and returns nothing.
    def __init__(self):
        self.vocab = {}
        self.id_to_token = {}
        self.merge_rank = {}
        self.pad_token_id = None
        self.eos_token_id = None
        self.padding_side = "right"
        self._cache = {}

    # It returns the vocabulary size.
    def __len__(self):
        return len(self.vocab)

    # It takes texts, target_vocab_size (target vocabulary size), sample_chars_per_doc (characters used per document) and verbose (print progress) as input, learns the BPE merges and vocabulary from the training texts and returns the trained tokenizer.
    def train(self, texts, target_vocab_size, sample_chars_per_doc=None, verbose=True):
        import time as _time
        t0 = _time.time()
        word_freq = Counter()
        for text in texts:
            if sample_chars_per_doc is not None:
                text = text[:sample_chars_per_doc]
            for m in _PRETOK_RE.finditer(text):
                sym = tuple(_BYTE_ENCODER[byte] for byte in m.group().encode("utf-8"))
                word_freq[sym] += 1

        self.vocab = {self.PAD_TOKEN: 0, self.EOS_TOKEN: 1}
        for i in range(256):
            self.vocab[_BYTE_ENCODER[i]] = len(self.vocab)
        self.pad_token_id = self.vocab[self.PAD_TOKEN]
        self.eos_token_id = self.vocab[self.EOS_TOKEN]

        words = {i: list(w) for i, w in enumerate(word_freq)}
        freqs = {i: c for i, (w, c) in enumerate(word_freq.items())}
        pair_counts = Counter()
        pair_to_words = _defaultdict(set)
        for wid, syms in words.items():
            f = freqs[wid]
            for i in range(len(syms) - 1):
                p = (syms[i], syms[i + 1])
                pair_counts[p] += f
                pair_to_words[p].add(wid)

        n_merges_needed = target_vocab_size - len(self.vocab)
        n_merges_done = 0
        while n_merges_done < n_merges_needed and pair_counts:
            max_freq = max(pair_counts.values())
            best = min(p for p, c in pair_counts.items() if c == max_freq)
            new_symbol = best[0] + best[1]
            self.merge_rank[best] = n_merges_done
            self.vocab[new_symbol] = len(self.vocab)
            for wid in list(pair_to_words.get(best, ())):
                syms, f = words[wid], freqs[wid]
                for i in range(len(syms) - 1):
                    p = (syms[i], syms[i + 1])
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        pair_counts.pop(p, None)
                    pair_to_words[p].discard(wid)
                merged, i = [], 0
                while i < len(syms):
                    if i < len(syms) - 1 and syms[i] == best[0] and syms[i + 1] == best[1]:
                        merged.append(new_symbol); i += 2
                    else:
                        merged.append(syms[i]); i += 1
                words[wid] = merged
                for i in range(len(merged) - 1):
                    p = (merged[i], merged[i + 1])
                    pair_counts[p] += f
                    pair_to_words[p].add(wid)
            pair_counts.pop(best, None)
            pair_to_words.pop(best, None)
            n_merges_done += 1

        self.id_to_token = {v: k for k, v in self.vocab.items()}
        if verbose:
            print(f"    BPE training: {n_merges_done} merges learned (target {n_merges_needed}) in "
                  f"{_time.time() - t0:.1f}s over {len(word_freq)} unique pretoken types "
                  f"-> final vocab size = {len(self.vocab)}")
        return self

    # It takes symbol_tuple (word as symbols) as input, applies the learned merges to one word and returns the word split into tokens (as a string).
    def _bpe(self, symbol_tuple):
        cached = self._cache.get(symbol_tuple)
        if cached is not None:
            return cached
        word = list(symbol_tuple)
        while len(word) > 1:
            best_rank, best_i = None, -1
            for i in range(len(word) - 1):
                r = self.merge_rank.get((word[i], word[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_i = r, i
            if best_rank is None:
                break
            word = word[:best_i] + [word[best_i] + word[best_i + 1]] + word[best_i + 2:]
        self._cache[symbol_tuple] = word
        return word

    # It takes text as input, converts text into a list of token ids and returns a list of token ids.
    def encode(self, text):
        ids = []
        for m in _PRETOK_RE.finditer(text):
            sym = tuple(_BYTE_ENCODER[byte] for byte in m.group().encode("utf-8"))
            for piece in self._bpe(sym):
                ids.append(self.vocab[piece])
        return ids

    # It takes ids (token ids) and skip_special_tokens (drop special tokens) as input, converts token ids back into text and returns text.
    def decode(self, ids, skip_special_tokens=True):
        special = {self.pad_token_id, self.eos_token_id}
        pieces = [self.id_to_token.get(int(i), "") for i in ids
                  if not (skip_special_tokens and int(i) in special)]
        raw = bytes(_BYTE_DECODER[c] for c in "".join(pieces) if c in _BYTE_DECODER)
        return raw.decode("utf-8", errors="replace")

    # It takes path (file path) as input, saves the tokenizer to a JSON file and returns nothing.
    def save(self, path):
        data = {
            "vocab": self.vocab,
            "merges": [list(pair) for pair, _rank in
                       sorted(self.merge_rank.items(), key=lambda kv: kv[1])],
            "pad_token_id": self.pad_token_id,
            "eos_token_id": self.eos_token_id,
            "padding_side": self.padding_side,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f)

    # It takes path (file path) as input, loads a tokenizer from a JSON file and returns a tokenizer.
    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls()
        tok.vocab = data["vocab"]
        tok.id_to_token = {int(v): k for k, v in tok.vocab.items()}
        tok.merge_rank = {tuple(pair): i for i, pair in enumerate(data["merges"])}
        tok.pad_token_id = data["pad_token_id"]
        tok.eos_token_id = data["eos_token_id"]
        tok.padding_side = data.get("padding_side", "right")
        return tok

    # It takes text, max_length (max length), truncation (cut long inputs), padding (padding option) and return_tensors (return tensors) as input, tokenizes one text or a list of texts, with optional cutting and padding and returns a dictionary with input_ids and attention_mask.
    def __call__(self, text, max_length=None, truncation=False, padding=None, return_tensors=None):
        is_batch = isinstance(text, (list, tuple))
        texts = list(text) if is_batch else [text]
        all_ids = [self.encode(t) for t in texts]
        if truncation and max_length is not None:
            all_ids = [ids[:max_length] for ids in all_ids]
        lens = [len(ids) for ids in all_ids]
        if padding == "max_length" and max_length is not None:
            pad_to = max_length
        elif padding is True or padding == "longest":
            pad_to = max(lens) if lens else 0
        else:
            pad_to = None
        if pad_to is not None:
            if self.padding_side == "right":
                attn = [[1] * L + [0] * (pad_to - L) for L in lens]
                all_ids = [ids + [self.pad_token_id] * (pad_to - len(ids)) for ids in all_ids]
            else:
                attn = [[0] * (pad_to - L) + [1] * L for L in lens]
                all_ids = [[self.pad_token_id] * (pad_to - len(ids)) + ids for ids in all_ids]
        else:
            attn = [[1] * L for L in lens]
        if return_tensors == "pt":
            return {"input_ids": torch.tensor(all_ids, dtype=torch.long),
                    "attention_mask": torch.tensor(attn, dtype=torch.long)}
        return {"input_ids": all_ids if is_batch else all_ids[0],
                "attention_mask": attn if is_batch else attn[0]}

class Tok:

    # It takes inner (inner tokenizer) as input, loads the tokenizer and returns nothing.
    def __init__(self, inner):
        self.inner = inner
        self.vocab_size = len(inner)
        self.pad_id = inner.pad_token_id
        self.eos_id = inner.eos_token_id
        self.bos_id = None

    # It takes text as input, converts text into token ids and returns a list of token ids.
    def encode(self, text):
        return self.inner.encode(text)

    # It takes ids (token ids) and skip_special_tokens (drop special tokens) as input, converts token ids back into text and returns text.
    def decode(self, ids, skip_special_tokens=True):
        return self.inner.decode(ids, skip_special_tokens)

    # It takes text as input, converts text into token ids and remember where each token is in the text and returns (token ids, character offsets).
    def encode_with_offsets(self, text):
        inner = self.inner
        ids, offs = [], []
        for m in _PRETOK_RE.finditer(text):
            s = m.group()
            b = s.encode("utf-8")
            byte2char = []
            for k, ch in enumerate(s):
                byte2char.extend([m.start() + k] * len(ch.encode("utf-8")))
            sym = tuple(_BYTE_ENCODER[x] for x in b)
            pos = 0
            for piece in inner._bpe(sym):
                n = len(piece)
                ids.append(inner.vocab[piece])
                offs.append((byte2char[pos], byte2char[pos + n - 1] + 1))
                pos += n
        return ids, offs

# It takes tok_path (tokenizer file path) as input, loads the tokenizer from its JSON file and returns a tokenizer.
def load_tokenizer(tok_path):
    return Tok(LegalByteBPETokenizer.load(tok_path))

# It takes path (file path) as input, turns an unzipped checkpoint folder back into a .pt file and returns (file path, temporary folder or None).
def _materialise_checkpoint_path(path):
    if not os.path.isdir(path):
        return path, None
    pkl = None
    for root, _d, files in os.walk(path):
        if "data.pkl" in files:
            pkl = root
            break
    if pkl is None:
        raise FileNotFoundError(f"{path} is a directory but contains no data.pkl")
    tmpdir = tempfile.mkdtemp(prefix="gp_ckpt_")
    out = os.path.join(tmpdir, "ckpt.pt")
    top = "archive"
    with zipfile.ZipFile(out, "w", compression=zipfile.ZIP_STORED) as zf:
        for root, _d, files in os.walk(pkl):
            for fn in sorted(files):
                full = os.path.join(root, fn)
                rel = os.path.relpath(full, pkl).replace(os.sep, "/")
                zf.write(full, arcname=f"{top}/{rel}")
    return out, tmpdir

# It takes obj (object) as input, gets the weights dictionary out of a saved checkpoint and returns weights dictionary.
def _extract_state_dict(obj):
    if isinstance(obj, nn.Module):
        return obj.state_dict()
    if isinstance(obj, dict):
        for k in ("best_state", "state_dict", "model_state_dict", "model_state", "model", "weights"):
            v = obj.get(k)
            if isinstance(v, nn.Module):
                return v.state_dict()
            if isinstance(v, dict) and any(torch.is_tensor(t) for t in v.values()):
                return v
        if any(torch.is_tensor(t) for t in obj.values()):
            return obj
    raise ValueError("could not find a state_dict in checkpoint")

# It takes sd (weights dictionary) as input, removes extra name prefixes from weight names and returns weights dictionary with clean names.
def _strip_prefixes(sd):
    out = {}
    for k, v in sd.items():
        while k.startswith("module."):
            k = k[len("module."):]
        out[k] = v
    return out

# It takes sd (weights dictionary) as input, works out the model settings from the saved weights and returns a dictionary of model settings.
def infer_config(sd):
    vocab, dim = sd["embed.tok.weight"].shape
    max_len = sd["embed.pos.weight"].shape[0]
    idx = {int(m.group(1)) for k in sd for m in [_re.match(r"blocks\.(\d+)\.", k)] if m}
    n_blocks = max(idx) + 1
    assert n_blocks % 2 == 0, "blocks should alternate GraphPropLayer/FeedForward"
    window = sd["blocks.0.rel_emb.weight"].shape[0]
    edge_dim = sd["blocks.0.edge_key.weight"].shape[0]
    return dict(vocab_size=vocab, dim=dim, n_layers=n_blocks // 2, window=window,
                edge_dim=edge_dim, max_len=max_len)

class Backbone:

    causal = True

    # It takes model and cfg (settings) as input, wraps the pre-trained model with simple helper methods and returns nothing.
    def __init__(self, model, cfg):
        self.model = model
        self.cfg = cfg
        self.d_model = cfg["dim"]
        self.max_len = cfg["max_len"]
        self.n_params = sum(p.numel() for p in model.parameters())
        self.bidirectional = False

    # It takes flag (on/off) as input, turns the causal mask off or on and returns nothing.
    def set_bidirectional(self, flag=True):
        self.bidirectional = bool(flag)
        self.causal = not self.bidirectional

    # It takes input_ids (token ids), attention_mask (mask of real tokens) and pos_offset (position offset) as input, returns the hidden vector of every token.
    def hidden(self, input_ids, attention_mask=None, pos_offset=0):
        m = self.model
        T = input_ids.size(1)
        assert T + pos_offset <= self.max_len, f"sequence {T}+{pos_offset} exceeds max_len={self.max_len}"
        x = m.drop(m.embed(input_ids, pos_offset=pos_offset))
        if self.bidirectional:
            valid = attention_mask.bool() if attention_mask is not None else None
            for blk in m.blocks:
                x = blk.forward_bidirectional(x, valid) if isinstance(blk, GraphPropLayer) else blk(x)
        else:
            for blk in m.blocks:
                x = blk(x)
        return m.norm_f(x)

    # It takes input_ids (token ids) and pos_offset (position offset) as input, returns next-token scores.
    def logits(self, input_ids, pos_offset=0):
        return self.model(input_ids, pos_offset=pos_offset)

    # It returns the list of model blocks.
    def block_modules(self):
        return list(self.model.blocks)

    # It returns the names of the linear layers inside the blocks.
    def linear_names(self):
        return [f"blocks.{n}" if n else "blocks"
                for n, mod in self.model.blocks.named_modules() if isinstance(mod, nn.Linear)]

# It takes ckpt_path (checkpoint path), device and dropout (dropout rate) as input, loads the pre-trained model from its checkpoint and returns the backbone ready to use.
def load_backbone(ckpt_path, device="cpu", dropout=0.1):
    path, tmp = _materialise_checkpoint_path(ckpt_path)
    try:
        obj = torch.load(path, weights_only=False, map_location="cpu")
    finally:
        if tmp:
            shutil.rmtree(tmp, ignore_errors=True)
    sd = _strip_prefixes(_extract_state_dict(obj))
    cfg = infer_config(sd)
    tied = ("lm_head.weight" not in sd) or torch.equal(sd["lm_head.weight"], sd["embed.tok.weight"])
    model = LegalGraphPropLM(cfg["vocab_size"], dim=cfg["dim"], n_layers=cfg["n_layers"],
                             window=cfg["window"], edge_dim=cfg["edge_dim"], max_len=cfg["max_len"],
                             dropout=dropout, tie=tied)
    if tied and "lm_head.weight" not in sd:
        sd = dict(sd); sd["lm_head.weight"] = sd["embed.tok.weight"]
    res = model.load_state_dict(sd, strict=False)
    if res.missing_keys or res.unexpected_keys:
        print("!" * 70)
        print("!!! STATE-DICT MISMATCH when loading GraphProp checkpoint !!!")
        print("missing:", res.missing_keys)
        print("unexpected:", res.unexpected_keys)
        print("!" * 70)
        raise RuntimeError("GraphProp checkpoint keys do not match model")
    model.to(device).eval()
    cfg["tied"] = tied
    return Backbone(model, cfg)

Overwriting arch_graphprop.py


In [12]:
# Import code, load data and build training config
PATHS_USED = [CKPT, TOK]

import sys, os, json, time, glob, random, zipfile
import numpy as np, torch
sys.path.insert(0, os.getcwd())
import rr_core as C, rr_peft as P
import arch_graphprop as A
import importlib
for _m in (C, P, A): importlib.reload(_m)
DECISIONS = dict(EPOCHS=EPOCHS, BS=BS, ACCUM=ACCUM, W=W, CTX=CTX, HEAD_LR=HEAD_LR, CLASS_POWER=CLASS_POWER, LABEL_SMOOTH=LABEL_SMOOTH, TAUS=TAUS, DEV_SEEDS=DEV_SEEDS, FULL_SEEDS=FULL_SEEDS, SWA_LAST=SWA_LAST, GRU_HIDDEN=GRU_HIDDEN, RARE_DUP=RARE_DUP, SNAP_EPOCHS=SNAP_EPOCHS, VIT_LAMS=VIT_LAMS, PEFT_LR=PEFT_LR, R=R, ALPHA=ALPHA, LORA_DROPOUT=LORA_DROPOUT, HEAD_DROPOUT=HEAD_DROPOUT)
missing = [k for k, v in DECISIONS.items() if v is None]
assert not missing, "Fill these in the Settings cell first: " + ", ".join(missing)
for p in PATHS_USED: assert p and os.path.exists(p), f"Path not found: {p!r} - attach the file and paste its path in the Settings cell"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
print("device:", DEVICE, "| GPUs:", torch.cuda.device_count())
os.makedirs(OUT, exist_ok=True); WORK = os.path.join(OUT, "_work"); os.makedirs(WORK, exist_ok=True)
RESUME_DIRS = [d for d in glob.glob("/kaggle/input/**/_work", recursive=True)]

C.ensure_phase3_data(repo=HF_DATASET)
raw = C.load_all(); labels = raw["labels"]; K = len(labels); label2i = {l: i for i, l in enumerate(labels)}
if SMOKE:
    for part in ("train", "dev", "test"): raw[part] = dict(list(raw[part].items())[:int(SMOKE_DOCS)])
print("labels:", labels)
for part in ("train", "dev", "test"):
    ds = raw[part]; print(f"{part:5s}: {len(ds)} documents ({sum(d['domain']=='CL' for d in ds.values())} CL / {sum(d['domain']=='IT' for d in ds.values())} IT), {sum(len(d['sents']) for d in ds.values())} sentences")
ids = [d["id"] for d in raw["test"].values()]
assert len(ids) == len(set(ids)), "a document id occurs in both CL_test and IT_test - read the evaluation script to see how the two files must be merged"
tr_lab = [l for d in raw["train"].values() for l in d["labels"]]
cnt = {l: tr_lab.count(l) for l in labels}; print("train label counts:", cnt)

tok = A.load_tokenizer(TOK)
bb0 = A.load_backbone(CKPT, device="cpu")
MAXTOK = int(bb0.max_len)
print("backbone parameters:", f"{bb0.n_params:,}", "| usable context:", MAXTOK, "tokens")
assert CTX == "sentence" or W + 2 <= MAXTOK, f"W={W} is longer than what this model was trained for ({MAXTOK} tokens)"
MAX_SENT = 200 if CTX == "window" else min(200, MAXTOK - 2)
data = dict(labels=labels, pad_id=tok.pad_id, raw=raw)
for part in ("train", "dev", "test"): data[part] = C.encode_docs(raw[part], tok, label2i, MAX_SENT) if part != "test" else C.encode_docs(raw[part], tok, {**label2i}, MAX_SENT)
lens = [sum(len(s) for s in e.ids) for e in data["train"].values()]
print("tokens per training document: mean", int(np.mean(lens)), "max", max(lens))
CFG = dict(epochs=EPOCHS, bs=BS, accum=ACCUM, W=W, ctx=CTX, class_power=CLASS_POWER, label_smooth=LABEL_SMOOTH, swa_last=SWA_LAST, rare_dup=RARE_DUP, snap_epochs=SNAP_EPOCHS, vit_lams=VIT_LAMS, warmup=0.06, wd=0.01, clip=1.0, save_every=1500)

device: cuda | GPUs: 2
Phase-3 files found locally: /kaggle/working/phase3_data/Phase_3_RR/Train
labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
train: 80 documents (40 CL / 40 IT), 16845 sentences
dev  : 10 documents (5 CL / 5 IT), 2142 sentences
test : 10 documents (5 CL / 5 IT), 2197 sentences
train label counts: {'Fact': 4098, 'Issue': 107, 'ArgumentPetitioner': 1613, 'ArgumentRespondent': 1348, 'Statute': 1099, 'Dissent': 284, 'PrecedentReliedUpon': 1945, 'PrecedentNotReliedUpon': 569, 'PrecedentOverruled': 3, 'RulingByLowerCourt': 874, 'RatioOfTheDecision': 4301, 'RulingByPresentCourt': 307, 'None': 297}
backbone parameters: 210,183,816 | usable context: 648 tokens
tokens per training document: mean 9376 max 82051


In [13]:
# Build the model with LoRA
NAME = "rr4_graphprop_lora"
# It takes seed (random seed) as input, builds the model with the chosen fine-tuning method and its learning rates and returns (model, learning-rate groups, info).
def build(seed):
    bb = A.load_backbone(CKPT, device="cpu")
    n = P.apply_linear_wrappers(bb, "lora", [n for n in bb.linear_names() if n.endswith('out_proj') or '.net.' in n], r=R, alpha=ALPHA, dropout=LORA_DROPOUT)
    model = C.RRModel(bb, K, CTX, GRU_HIDDEN, HEAD_DROPOUT)
    return model, C.make_groups(model, HEAD_LR, PEFT_LR), f"lora: {n} linear layers wrapped, r={R}, alpha={ALPHA}"

In [14]:
# Run training and evaluation
t_start = time.time()
summary = C.run_experiment(NAME, build, CFG, data, OUT, WORK, resume_dirs=RESUME_DIRS, taus=TAUS, dev_seeds=DEV_SEEDS, full_seeds=FULL_SEEDS, stages=STAGES)
print("total time", C._hms(time.time() - t_start))


===== rr4_graphprop_lora: stage 'dev' (train -> dev), seeds [0] =====
  seed 0: trainable parameters 7,480,845 of 217,664,661 (3.44 %)  lora: 24 linear layers wrapped, r=16, alpha=32
  micro-batches/epoch 901 (bs 4 x accum 4), optimizer steps/epoch 225, total 3375; rare labels [1, 5, 8, 11, 12]
  resumed from rr4_graphprop_lora_dev_s0.resume.pt at epoch 15/15


epoch 15/15:   0%|          | 0/225 [00:00<?, ?it/s]

  epoch 15  train_ce 0.7491  secs 233.6194  val_macro_F1 0.3652  val_CL 0.2851  val_IT 0.3988  | epoch 15/15, run ETA 0h00m  [overall 47% | elapsed 0h04m | ETA 1h07m]
  dev Macro-F1 [avg   tau 0]  all 0.3652  CL 0.2851  IT 0.3988  (all 13 labels: 0.3371)
  dev Macro-F1 [avg   tau 0 viterbi 0.5]  all 0.3882  CL 0.3413  IT 0.4125  (all 13 labels: 0.3583)
  dev Macro-F1 [avg   tau 0.3]  all 0.1934  CL 0.1478  IT 0.2032  (all 13 labels: 0.1785)
  dev Macro-F1 [avg   tau 0.3 viterbi 0.5]  all 0.2521  CL 0.2182  IT 0.2524  (all 13 labels: 0.2327)
  dev Macro-F1 [avg   tau 0.6]  all 0.0619  CL 0.0352  IT 0.0661  (all 13 labels: 0.0572)
  dev Macro-F1 [avg   tau 0.6 viterbi 0.5]  all 0.0873  CL 0.1009  IT 0.0554  (all 13 labels: 0.0806)
  dev Macro-F1 [avg   tau 1]  all 0.0098  CL 0.0000  IT 0.0119  (all 13 labels: 0.0090)
  dev Macro-F1 [avg   tau 1 viterbi 0.5]  all 0.0000  CL 0.0000  IT 0.0000  (all 13 labels: 0.0000)
    per label (first tau, no viterbi):  Fact F1 0.61 (n=444)  Issue F1 0.

epoch 1/15:   0%|          | 0/253 [00:00<?, ?it/s]

  epoch 1  train_ce 2.1480  secs 263.3162  | epoch 1/15, run ETA 1h01m  [overall 51% | elapsed 0h09m | ETA 1h08m]


epoch 2/15:   0%|          | 0/253 [00:00<?, ?it/s]

... (23 outputs left out)

epoch 14/15:   0%|          | 0/253 [00:00<?, ?it/s]

  epoch 14  train_ce 0.7699  secs 263.2077  | epoch 14/15, run ETA 0h04m  [overall 96% | elapsed 1h06m | ETA 0h04m]


epoch 15/15:   0%|          | 0/254 [00:00<?, ?it/s]

  epoch 15  train_ce 0.7669  secs 264.0004  | epoch 15/15, run ETA 0h00m  [overall 100% | elapsed 1h10m | ETA 0h00m]
    wrote rr4_graphprop_lora__full_avg__tau0_submission.zip
    wrote rr4_graphprop_lora__full_avg__tau0_vit0.5_submission.zip
    wrote rr4_graphprop_lora__full_avg__tau0.3_submission.zip
    wrote rr4_graphprop_lora__full_avg__tau0.3_vit0.5_submission.zip
    wrote rr4_graphprop_lora__full_avg__tau0.6_submission.zip
    wrote rr4_graphprop_lora__full_avg__tau0.6_vit0.5_submission.zip
    wrote rr4_graphprop_lora__full_avg__tau1_submission.zip
    wrote rr4_graphprop_lora__full_avg__tau1_vit0.5_submission.zip
    wrote rr4_graphprop_lora__full_final__tau0_submission.zip
    wrote rr4_graphprop_lora__full_final__tau0_vit0.5_submission.zip
    wrote rr4_graphprop_lora__full_final__tau0.3_submission.zip
    wrote rr4_graphprop_lora__full_final__tau0.3_vit0.5_submission.zip
    wrote rr4_graphprop_lora__full_final__tau0.6_submission.zip
    wrote rr4_graphprop_lora__full_fi

In [15]:
# Show dev scores of all variants
rows = []
for stg, info in summary["stages"].items():
    for key, sc in info.get("scores", {}).items():
        rows.append((sc["all"]["macro_f1"], key, {d: round(v["macro_f1"], 4) for d, v in sc.items()}))
for f1, key, per in sorted(rows, reverse=True): print(f"dev Macro-F1 {f1:.4f}   {key:14s}  {per}")
print("\nSubmission zips (each contains only prediction.json at the zip root):")
for f in sorted(glob.glob(os.path.join(OUT, "*_submission.zip"))): print("  ", os.path.basename(f), f"{os.path.getsize(f)/1e6:.1f} MB")
print("\nThe '__full_' zips come from the model retrained on ALL documents (train + dev): these are the ones to submit. "
      "Pick the tau (and weights kind) that was best on the 'dev' rows; the dev rows themselves were produced without ever seeing the dev documents.")

dev Macro-F1 0.4132   best_tau0_vit0.5  {'all': 0.4132, 'CL': 0.3445, 'IT': 0.467}
dev Macro-F1 0.3882   final_tau0_vit0.5  {'all': 0.3882, 'CL': 0.3413, 'IT': 0.4125}
dev Macro-F1 0.3882   avg_tau0_vit0.5  {'all': 0.3882, 'CL': 0.3413, 'IT': 0.4125}
dev Macro-F1 0.3818   best_tau0       {'all': 0.3818, 'CL': 0.3146, 'IT': 0.4047}
dev Macro-F1 0.3652   final_tau0      {'all': 0.3652, 'CL': 0.2851, 'IT': 0.3988}
dev Macro-F1 0.3652   avg_tau0        {'all': 0.3652, 'CL': 0.2851, 'IT': 0.3988}
dev Macro-F1 0.2725   best_tau0.3_vit0.5  {'all': 0.2725, 'CL': 0.2293, 'IT': 0.2882}
dev Macro-F1 0.2521   final_tau0.3_vit0.5  {'all': 0.2521, 'CL': 0.2182, 'IT': 0.2524}
dev Macro-F1 0.2521   avg_tau0.3_vit0.5  {'all': 0.2521, 'CL': 0.2182, 'IT': 0.2524}
dev Macro-F1 0.2264   best_tau0.3     {'all': 0.2264, 'CL': 0.1677, 'IT': 0.2571}
dev Macro-F1 0.1934   final_tau0.3    {'all': 0.1934, 'CL': 0.1478, 'IT': 0.2032}
dev Macro-F1 0.1934   avg_tau0.3      {'all': 0.1934, 'CL': 0.1478, 'IT': 0.2032}

In [16]:
# Zip all outputs
import shutil
zp = os.path.join(OUT, "RR4_GraphProp_LoRA_ALL_outputs.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(os.path.join(OUT, "*"))):
        if os.path.isfile(f) and not f.endswith("_ALL_outputs.zip"): z.write(f, os.path.basename(f))
    for f in glob.glob(os.path.join(WORK, "*.npz")): z.write(f, "_work/" + os.path.basename(f))
print(zp, f"{os.path.getsize(zp)/1e6:.1f} MB")

/kaggle/working/RR4_GraphProp_LoRA_ALL_outputs.zip 1.4 MB
